# 📖 Semana 13 · Unidad 4 — Symbol Tables

## Información del Curso

| Aspecto | Detalle |
|--------|--------|
| **Universidad** | Universidad de Talca, Chile |
| **Carrera** | Ingeniería Civil en Informática |
| **Semestre** | 2°-3° año |
| **Curso** | Algoritmos y Estructuras de Datos |
| **Docente** | PhD. César Astudillo |
| **Clase** | Semana 13 · Unidad 4 — Symbol Tables |
| **Duración** | 90 minutos |

---
> 🎯 *Este notebook está diseñado para ser ejecutado en clase de forma interactiva.*  
> *Ejecuta las celdas en orden de arriba hacia abajo.*

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import time
import random
from IPython.display import display, HTML
print("✅ Dependencias cargadas correctamente")

## 🎯 Objetivos de Aprendizaje

Al finalizar esta sesión, el estudiante será capaz de:

1. **Definir** el ADT Symbol Table y su API (put, get, delete, contains, size).
2. **Implementar** Sequential Search sobre lista enlazada desordenada y analizar su complejidad.
3. **Implementar** Binary Search sobre arreglo ordenado, identificando `rank` como operación central.
4. **Utilizar** la Ordered ST API (min, max, floor, ceiling, rank, select) con ejemplos concretos.
5. **Comparar** ambas implementaciones en términos de costo de operaciones.

---
# Sección 1: ¿Qué es una Symbol Table? (10 minutos)

## El problema que resuelve

Una **Symbol Table** (tabla de símbolos, diccionario, mapa) es una estructura de datos que asocia una **clave** (*key*) con un **valor** (*value*).

La operación fundamental es doble:
- `put(key, value)` → insertar o actualizar
- `get(key)` → recuperar el valor asociado a la clave

### Ejemplos del mundo real

| Aplicación | Clave | Valor |
|-----------|-------|-------|
| Guía telefónica | Nombre | Teléfono |
| DNS | Nombre de dominio | Dirección IP |
| Diccionario | Palabra | Definición |
| Compilador | Variable | Tipo y dirección de memoria |
| Caché web | URL | Contenido de la página |
| Índice de libro | Término | Páginas donde aparece |

> 🎙️ **[PAUSA PROFESOR]** *"¿Qué otras aplicaciones de su vida cotidiana usan una asociación clave-valor?"*

## La API básica

```python
class ST:
    def put(key, value)   # asociar value a key (sobreescribe si ya existe)
    def get(key)          # retornar el valor asociado a key (None si no existe)
    def delete(key)       # eliminar key y su valor
    def contains(key)     # ¿existe key en la tabla?
    def isEmpty()         # ¿está vacía la tabla?
    def size()            # número de pares clave-valor
    def keys()            # iterable con todas las claves
```

## Convenciones importantes (Sedgewick)

| Convención | Razón |
|-----------|-------|
| No se permiten claves duplicadas | Un `put` sobre clave existente **sobreescribe** el valor |
| No se permiten claves `None` | Simplifica la interfaz y evita errores silenciosos |
| `put(key, None)` equivale a `delete(key)` | Convención de Sedgewick para simplificar |
| `get(key)` retorna `None` si la clave no existe | No lanza excepción por clave ausente |

```python
# contains se puede implementar usando get:
def contains(key):
    return get(key) is not None
```

---
# Sección 2: Sequential Search — Lista Desordenada (20 minutos)

## Implementación con lista enlazada

La implementación más simple: mantener una lista de nodos `(key, value)` **sin ningún orden particular**.

```
head → [S|0] → [E|1] → [A|8] → [R|3] → [C|4] → [H|5] → None
```

**`get(key)`:** recorrer la lista comparando claves hasta encontrar o llegar al final.

**`put(key, value)`:**
1. Buscar la clave en la lista (por si ya existe).
2. Si existe: actualizar el valor.
3. Si no existe: insertar un nuevo nodo al principio.

## Análisis de complejidad

| Operación | Caso promedio | Peor caso | Observación |
|-----------|-------------|-----------|-------------|
| `get` (éxito) | ~N/2 comp. | N comp. | Recorre hasta encontrar |
| `get` (fracaso) | N comp. | N comp. | Debe recorrer toda la lista |
| `put` (nueva clave) | N comp. | N comp. | Buscar + insertar al principio |
| `put` (clave existente) | ~N/2 comp. | N comp. | Buscar y actualizar |

> 📌 En tablas grandes, la búsqueda secuencial es prohibitivamente lenta. Pero para N pequeño (< 10 elementos), suele ser la opción más eficiente en la práctica por su simplicidad.

In [ ]:
class Node:
    """Nodo de la lista enlazada."""
    def __init__(self, key, value, next_node=None):
        self.key   = key
        self.value = value
        self.next  = next_node


class SequentialST:
    """
    Symbol Table con búsqueda secuencial.
    Implementación: lista enlazada desordenada.
    Complejidad: get O(N), put O(N).
    """

    def __init__(self):
        self._head = None   # primer nodo de la lista
        self._n    = 0      # número de pares

    def size(self):
        return self._n

    def isEmpty(self):
        return self._n == 0

    def get(self, key):
        """Búsqueda secuencial — O(N)."""
        if key is None:
            raise ValueError("key no puede ser None")
        node = self._head
        while node is not None:
            if node.key == key:
                return node.value   # encontrado
            node = node.next
        return None                 # no encontrado

    def put(self, key, value):
        """Insertar o actualizar — O(N)."""
        if key is None:
            raise ValueError("key no puede ser None")
        if value is None:           # convención: None = delete
            self.delete(key)
            return
        # Buscar si la clave ya existe
        node = self._head
        while node is not None:
            if node.key == key:
                node.value = value  # actualizar
                return
            node = node.next
        # No existe: insertar al principio
        self._head = Node(key, value, self._head)
        self._n += 1

    def contains(self, key):
        return self.get(key) is not None

    def delete(self, key):
        """Eliminar clave — O(N)."""
        if key is None:
            raise ValueError("key no puede ser None")
        # Caso: clave en el primer nodo
        if self._head is not None and self._head.key == key:
            self._head = self._head.next
            self._n -= 1
            return
        # Caso general: buscar el nodo anterior
        prev = self._head
        while prev is not None and prev.next is not None:
            if prev.next.key == key:
                prev.next = prev.next.next
                self._n -= 1
                return
            prev = prev.next

    def keys(self):
        """Iterable con todas las claves."""
        node = self._head
        while node is not None:
            yield node.key
            node = node.next

    def __repr__(self):
        items = [(k, self.get(k)) for k in self.keys()]
        return "SequentialST(" + ", ".join(f"{k}:{v}" for k, v in items) + ")"


# ── Demo ────────────────────────────────────────────
st = SequentialST()

# Insertar pares de ejemplo (Sedgewick usa letras como claves)
pares = [('S', 0), ('E', 1), ('A', 8), ('R', 3), ('C', 4),
         ('H', 5), ('E', 12), ('X', 7), ('A', 0), ('M', 9)]

print("Insertando pares (key, value):")
print("-" * 45)
for k, v in pares:
    st.put(k, v)
    print(f"  put('{k}', {v:2d})  →  size = {st.size()}")

print(f"\nEstado final: {st}")
print(f"\nBúsquedas:")
for k in ['A', 'E', 'Z']:
    print(f"  get('{k}') = {st.get(k)}")

In [ ]:
# Visualización: comparaciones requeridas por get
class SequentialST_Conteo(SequentialST):
    """Versión instrumentada que cuenta comparaciones."""
    def __init__(self):
        super().__init__()
        self.comparaciones = 0

    def get(self, key):
        node = self._head
        while node is not None:
            self.comparaciones += 1
            if node.key == key:
                return node.value
            node = node.next
        return None

ns = [100, 500, 1000, 2000, 5000, 10000]
comp_exito   = []
comp_fracaso = []

for n in ns:
    claves = list(range(n))
    st_c = SequentialST_Conteo()
    for k in claves:
        st_c.put(k, k * 2)

    # Búsqueda con éxito: 100 búsquedas aleatorias
    st_c.comparaciones = 0
    muestra = random.sample(claves, 100)
    for k in muestra:
        st_c.get(k)
    comp_exito.append(st_c.comparaciones / 100)

    # Búsqueda sin éxito: claves que no existen
    st_c.comparaciones = 0
    for k in range(n, n + 100):
        st_c.get(k)
    comp_fracaso.append(st_c.comparaciones / 100)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(ns, comp_exito,   'bo-', label='Búsqueda con éxito (~N/2)', linewidth=2)
ax.plot(ns, comp_fracaso, 'rs-', label='Búsqueda sin éxito (~N)',   linewidth=2)
ax.plot(ns, [n/2 for n in ns], 'b--', alpha=0.5, label='N/2 teórico')
ax.plot(ns, ns,                'r--', alpha=0.5, label='N teórico')
ax.set_xlabel('n (tamaño de la tabla)', fontsize=12)
ax.set_ylabel('Comparaciones promedio', fontsize=12)
ax.set_title('Sequential Search — Comparaciones vs N', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"{'n':>8} {'Éxito':>12} {'N/2':>10} {'Fracaso':>12} {'N':>10}")
print("-" * 55)
for i, n in enumerate(ns):
    print(f"{n:>8} {comp_exito[i]:>12.1f} {n/2:>10.1f} {comp_fracaso[i]:>12.1f} {n:>10}")

---
# Sección 3: Binary Search — Arreglo Ordenado (25 minutos)

## La idea central: mantener el orden

Si en lugar de una lista desordenada usamos **dos arreglos paralelos ordenados por clave**, podemos aplicar **búsqueda binaria** para `get`, reduciendo el costo de O(N) a O(log N).

```
índice:    0    1    2    3    4    5    6    7
keys[]:   'A'  'C'  'E'  'H'  'M'  'R'  'S'  'X'
vals[]:    0    4   12    5    9    3    0    7
```

## La operación `rank` — el núcleo de todo

`rank(key)` retorna el **número de claves en la tabla que son menores que `key`**.

- Si `key` **existe** en la tabla: `keys[rank(key)] == key`.
- Si `key` **no existe**: `rank(key)` es la posición donde debería insertarse.

```
rank('E') = 2   → keys[2] = 'E'   ✓ (existe, en posición 2)
rank('G') = 3   → keys[3] = 'H'  ≠ 'G'  (no existe, insertaría en pos 3)
rank('Z') = 8   → más allá del arreglo (no existe)
```

## Implementación de rank con búsqueda binaria

```python
def rank(key):
    lo, hi = 0, n - 1
    while lo <= hi:
        mid = (lo + hi) // 2
        if   key < keys[mid]: hi = mid - 1   # buscar en mitad izquierda
        elif key > keys[mid]: lo = mid + 1   # buscar en mitad derecha
        else:                 return mid      # encontrado
    return lo   # no encontrado: posición de inserción
```

## Cómo get y put usan rank

```python
def get(key):
    i = rank(key)
    if i < n and keys[i] == key:
        return vals[i]   # existe
    return None          # no existe

def put(key, value):
    i = rank(key)
    if i < n and keys[i] == key:
        vals[i] = value    # actualizar: O(1) después de rank
        return
    # Insertar en posición i: desplazar todo lo que está a la derecha
    # → O(N) por el desplazamiento
    ...
```

> 🎙️ **[PAUSA PROFESOR]** *"¿Por qué `put` sigue siendo O(N) aunque usemos búsqueda binaria para encontrar la posición?"*  
> *Respuesta esperada: encontrar la posición es O(log N), pero desplazar los elementos del arreglo para hacer espacio es O(N).*

In [ ]:
class BinarySearchST:
    """
    Symbol Table con búsqueda binaria sobre arreglo ordenado.
    Complejidad: get O(log N), put O(N), operaciones de orden O(log N) o O(1).
    """

    def __init__(self, capacity=10):
        self._keys = [None] * capacity
        self._vals = [None] * capacity
        self._n    = 0

    def size(self):
        return self._n

    def isEmpty(self):
        return self._n == 0

    # ──────────────────────────────────────────────
    # rank: núcleo de la implementación
    # ──────────────────────────────────────────────
    def rank(self, key):
        """
        Retorna el número de claves en la tabla menores que key.
        Implementación: búsqueda binaria → O(log N).

        Invariante al terminar:
          - Si key existe:     keys[rank(key)] == key
          - Si key no existe:  rank(key) es la posición de inserción
        """
        lo, hi = 0, self._n - 1
        while lo <= hi:
            mid = (lo + hi) // 2
            if   key < self._keys[mid]: hi = mid - 1
            elif key > self._keys[mid]: lo = mid + 1
            else:                       return mid
        return lo

    def get(self, key):
        """Búsqueda binaria — O(log N)."""
        if key is None:
            raise ValueError("key no puede ser None")
        i = self.rank(key)
        if i < self._n and self._keys[i] == key:
            return self._vals[i]
        return None

    def put(self, key, value):
        """Insertar o actualizar — get O(log N), desplazamiento O(N)."""
        if key is None:
            raise ValueError("key no puede ser None")
        if value is None:
            self.delete(key)
            return

        i = self.rank(key)

        # Si ya existe: solo actualizar el valor
        if i < self._n and self._keys[i] == key:
            self._vals[i] = value
            return

        # Si el arreglo está lleno: expandir
        if self._n == len(self._keys):
            self._resize(2 * len(self._keys))

        # Desplazar elementos a la derecha para hacer espacio en posición i
        for j in range(self._n, i, -1):
            self._keys[j] = self._keys[j - 1]
            self._vals[j] = self._vals[j - 1]

        self._keys[i] = key
        self._vals[i] = value
        self._n += 1

    def contains(self, key):
        return self.get(key) is not None

    def delete(self, key):
        """Eliminar clave — O(N) por el desplazamiento."""
        if not self.contains(key):
            return
        i = self.rank(key)
        for j in range(i, self._n - 1):
            self._keys[j] = self._keys[j + 1]
            self._vals[j] = self._vals[j + 1]
        self._n -= 1
        self._keys[self._n] = None
        self._vals[self._n] = None

    def _resize(self, new_cap):
        new_keys = [None] * new_cap
        new_vals = [None] * new_cap
        for i in range(self._n):
            new_keys[i] = self._keys[i]
            new_vals[i] = self._vals[i]
        self._keys = new_keys
        self._vals = new_vals

    def keys(self):
        """Todas las claves en orden."""
        return self._keys[:self._n]

    def __repr__(self):
        pares = [(self._keys[i], self._vals[i]) for i in range(self._n)]
        return "BinarySearchST(" + ", ".join(f"{k}:{v}" for k, v in pares) + ")"


# ── Demo ────────────────────────────────────────────
bst = BinarySearchST()

pares = [('S', 0), ('E', 1), ('A', 8), ('R', 3), ('C', 4),
         ('H', 5), ('E', 12), ('X', 7), ('A', 0), ('M', 9)]

print("Insertando pares (key, value):")
print("-" * 55)
for k, v in pares:
    bst.put(k, v)
    print(f"  put('{k}', {v:2d})  →  keys = {bst.keys()}")

print(f"\nEstado final: {bst}")
print(f"\nBúsquedas con rank:")
for k in ['A', 'E', 'G', 'Z']:
    r = bst.rank(k)
    v = bst.get(k)
    print(f"  rank('{k}') = {r}  →  get('{k}') = {v}")

In [ ]:
# Visualización: tracing de rank
def rank_verbose(keys, n, key):
    """Versión pedagógica de rank que muestra cada paso."""
    print(f"rank('{key}') sobre {keys[:n]}")
    lo, hi = 0, n - 1
    paso = 0
    while lo <= hi:
        mid = (lo + hi) // 2
        intervalo = keys[lo:hi+1]
        print(f"  Paso {paso}: lo={lo}, hi={hi}, mid={mid}  "
              f"keys[mid]='{keys[mid]}'  intervalo={intervalo}")
        if   key < keys[mid]:
            print(f"         '{key}' < '{keys[mid]}' → buscar izquierda")
            hi = mid - 1
        elif key > keys[mid]:
            print(f"         '{key}' > '{keys[mid]}' → buscar derecha")
            lo = mid + 1
        else:
            print(f"         '{key}' == '{keys[mid]}' → ENCONTRADO en posición {mid}")
            return mid
        paso += 1
    print(f"  No encontrado. Posición de inserción: {lo}")
    return lo

keys_ejemplo = ['A', 'C', 'E', 'H', 'M', 'R', 'S', 'X']
n_ej = len(keys_ejemplo)

print("=" * 50)
rank_verbose(keys_ejemplo, n_ej, 'H')  # clave que existe
print("=" * 50)
rank_verbose(keys_ejemplo, n_ej, 'G')  # clave que NO existe
print("=" * 50)
rank_verbose(keys_ejemplo, n_ej, 'Z')  # mayor que todas

---
# Sección 4: Ordered ST API — El Ejemplo de los Trenes (25 minutos)

## ¿Por qué una Ordered ST?

Cuando las claves son **comparables** (números, fechas, strings), podemos soportar consultas de **orden** que no tiene sentido en una tabla desordenada:

```python
class OrderedST(ST):
    def min()              # clave mínima
    def max()              # clave máxima
    def floor(key)         # mayor clave ≤ key
    def ceiling(key)       # menor clave ≥ key
    def rank(key)          # número de claves < key
    def select(k)          # clave de rango k (la k-ésima clave)
    def keys(lo, hi)       # claves en el rango [lo, hi]
    def size(lo, hi)       # número de claves en [lo, hi]
    def deleteMin()        # eliminar la clave mínima
    def deleteMax()        # eliminar la clave máxima
```

## El ejemplo de los trenes

Considera una tabla donde cada **clave es la hora de llegada** de un tren (como string `"HH:MM"`) y el **valor es el nombre de la estación** de origen.

```
Hora     Estación
06:10  → Constitución
07:45  → Lo Espejo
08:30  → Alameda
09:15  → Pudahuel
10:00  → Maipu
11:30  → Cerrillos
13:00  → San Bernardo
15:20  → Buin
```

Con la Ordered ST API podemos responder:

| Consulta | Operación | Resultado |
|---------|-----------|----------|
| ¿Primer tren del día? | `min()` | `"06:10"` → Constitución |
| ¿Último tren del día? | `max()` | `"15:20"` → Buin |
| ¿Último tren antes de las 09:00? | `floor("09:00")` | `"08:30"` → Alameda |
| ¿Próximo tren desde las 09:00? | `ceiling("09:00")` | `"09:15"` → Pudahuel |
| ¿Cuántos trenes llegan antes de las 10:00? | `rank("10:00")` | `4` |
| ¿Cuál es el 3er tren del día? | `select(2)` | `"08:30"` → Alameda |
| ¿Trenes entre las 08:00 y las 12:00? | `keys("08:00", "12:00")` | `["08:30", "09:15", "10:00", "11:30"]` |

> 🎙️ **[PAUSA PROFESOR]** *"¿Cuál es la diferencia entre `floor` y `ceiling` cuando la clave exacta SÍ existe en la tabla?"*  
> *Respuesta: tanto `floor(key)` como `ceiling(key)` retornan `key` mismo cuando existe.*

In [ ]:
class OrderedST(BinarySearchST):
    """
    Ordered Symbol Table — extiende BinarySearchST con operaciones de orden.
    Todas las operaciones de orden se apoyan en rank(), que ya es O(log N).
    """

    def min(self):
        """Clave mínima — O(1)."""
        if self.isEmpty():
            raise IndexError("Tabla vacía")
        return self._keys[0]

    def max(self):
        """Clave máxima — O(1)."""
        if self.isEmpty():
            raise IndexError("Tabla vacía")
        return self._keys[self._n - 1]

    def select(self, k):
        """
        Clave de rango k (la k-ésima más pequeña, 0-based) — O(1).
        select(0) = min(), select(n-1) = max()
        """
        if k < 0 or k >= self._n:
            raise IndexError(f"rango {k} fuera de [0, {self._n-1}]")
        return self._keys[k]

    def floor(self, key):
        """
        Mayor clave ≤ key — O(log N).
        Si key existe en la tabla: retorna key.
        Si key no existe: retorna la clave más grande menor que key.
        Retorna None si todas las claves son mayores que key.
        """
        i = self.rank(key)
        # ¿Existe key exactamente?
        if i < self._n and self._keys[i] == key:
            return key
        # La clave más grande menor que key está en posición i-1
        if i == 0:
            return None    # no hay nada a la izquierda
        return self._keys[i - 1]

    def ceiling(self, key):
        """
        Menor clave ≥ key — O(log N).
        Si key existe en la tabla: retorna key.
        Si key no existe: retorna la clave más pequeña mayor que key.
        Retorna None si todas las claves son menores que key.
        """
        i = self.rank(key)
        if i >= self._n:
            return None    # key es mayor que todas las claves
        return self._keys[i]  # incluye el caso key == keys[i]

    def keys_range(self, lo, hi):
        """Claves en el rango [lo, hi] en orden — O(log N + k) donde k = resultado."""
        result = []
        i = self.rank(lo)
        while i < self._n and self._keys[i] <= hi:
            result.append(self._keys[i])
            i += 1
        return result

    def size_range(self, lo, hi):
        """Número de claves en [lo, hi] — O(log N)."""
        if lo > hi:
            return 0
        if self.contains(hi):
            return self.rank(hi) - self.rank(lo) + 1
        return self.rank(hi) - self.rank(lo)


# ── Demo: el horario de trenes ────────────────────────────────
horario = OrderedST(capacity=20)

trenes = [
    ('06:10', 'Constitución'),
    ('07:45', 'Lo Espejo'),
    ('08:30', 'Alameda'),
    ('09:15', 'Pudahuel'),
    ('10:00', 'Maipu'),
    ('11:30', 'Cerrillos'),
    ('13:00', 'San Bernardo'),
    ('15:20', 'Buin'),
]

for hora, estacion in trenes:
    horario.put(hora, estacion)

print("Horario de trenes cargado:")
for hora in horario.keys():
    print(f"  {hora}  →  {horario.get(hora)}")

print()
print("Consultas con Ordered ST API:")
print("-" * 55)

# min y max
print(f"  min()           = {horario.min()}  → {horario.get(horario.min())}")
print(f"  max()           = {horario.max()}  → {horario.get(horario.max())}")

# floor y ceiling
for hora_consulta in ['09:00', '09:15', '06:00', '16:00']:
    f = horario.floor(hora_consulta)
    c = horario.ceiling(hora_consulta)
    est_f = horario.get(f) if f else None
    est_c = horario.get(c) if c else None
    print(f"  floor('{hora_consulta}') = {f}  → {est_f}")
    print(f"  ceiling('{hora_consulta}') = {c}  → {est_c}")
    print()

# rank y select
print(f"  rank('10:00')   = {horario.rank('10:00')}  (trenes antes de las 10:00)")
print(f"  select(2)       = {horario.select(2)}  → {horario.get(horario.select(2))}  (3er tren)")

# rango
rango = horario.keys_range('08:00', '12:00')
print(f"  keys('08:00','12:00') = {rango}")
print(f"  size('08:00','12:00') = {horario.size_range('08:00', '12:00')}")

In [ ]:
# Visualización: floor y ceiling sobre la línea de tiempo
def visualizar_floor_ceiling(horario, consulta):
    horas = horario.keys()
    posiciones = list(range(len(horas)))

    fig, ax = plt.subplots(figsize=(13, 3))

    # Línea de tiempo
    ax.plot(posiciones, [0]*len(posiciones), 'k-', linewidth=2, zorder=1)

    # Marcar cada tren
    for i, hora in enumerate(horas):
        ax.plot(i, 0, 'o', color='steelblue', markersize=14, zorder=2)
        ax.text(i, 0.15, hora, ha='center', va='bottom', fontsize=8, rotation=45)
        ax.text(i, -0.2, horario.get(hora), ha='center', va='top', fontsize=7, color='gray')

    # Marcar floor y ceiling de la consulta
    f = horario.floor(consulta)
    c = horario.ceiling(consulta)

    # Posición relativa de la consulta
    r = horario.rank(consulta)
    consulta_x = r - 0.5 if r > 0 else -0.5
    if f == consulta:   # existe exactamente
        consulta_x = r

    ax.axvline(x=consulta_x, color='red', linewidth=2, linestyle='--', alpha=0.7, label=f'consulta: "{consulta}"')

    if f is not None:
        fi = horas.index(f)
        ax.plot(fi, 0, 's', color='green', markersize=18, zorder=3, alpha=0.7, label=f'floor = "{f}"')

    if c is not None:
        ci = horas.index(c)
        ax.plot(ci, 0, 'D', color='orange', markersize=18, zorder=3, alpha=0.7, label=f'ceiling = "{c}"')

    ax.set_xlim(-0.8, len(horas) - 0.2)
    ax.set_ylim(-0.5, 0.7)
    ax.axis('off')
    ax.legend(loc='upper left', fontsize=10)
    ax.set_title(f'floor("{consulta}") y ceiling("{consulta}") sobre el horario de trenes',
                 fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.show()

visualizar_floor_ceiling(horario, '09:00')
visualizar_floor_ceiling(horario, '09:15')

---
# Sección 5: Comparación de Costos (10 minutos)

## Resumen de complejidades

| Operación | Sequential Search | Binary Search | Observación |
|-----------|:-----------------:|:-------------:|-------------|
| `get` | O(N) | **O(log N)** | BS gana gracias a `rank` |
| `put` | O(N) | O(N) | BS pierde por el desplazamiento |
| `contains` | O(N) | O(log N) | Usa `get` internamente |
| `delete` | O(N) | O(N) | Ambas desplazan o recorren |
| `min` / `max` | O(N) | **O(1)** | BS guarda claves ordenadas |
| `floor` / `ceiling` | — | **O(log N)** | Solo disponible en Ordered ST |
| `rank` / `select` | — | **O(log N)** / O(1) | Solo disponible en Ordered ST |

## El problema que persiste

Binary Search logra **get en O(log N)**, pero **put sigue siendo O(N)** por el desplazamiento del arreglo.

Para una aplicación con muchas inserciones (índice web, compilador, base de datos), esto es inaceptable.

> 🎙️ **[PREGUNTA DE CIERRE]** *"¿Podemos lograr get Y put ambos en O(log N) con una sola estructura? ¿Qué estructura conocen que tenga esta propiedad?"*

*(Respuesta en las próximas clases: Binary Search Trees y sus variantes balanceadas)*

## ¿Cuándo usar cada implementación?

| Escenario | Recomendación |
|-----------|---------------|
| N muy pequeño (< 10) | Sequential Search (simplicidad) |
| Muchas búsquedas, pocas inserciones | Binary Search |
| Necesito `floor`/`ceiling`/`rank` | Binary Search (Ordered ST) |
| Muchas inserciones y búsquedas | BST o Hash Table (próximas clases) |

In [ ]:
# Comparación empírica: tiempos de get y put
import time

ns = [100, 500, 1000, 2000, 5000]

tiempos_seq_put  = []
tiempos_seq_get  = []
tiempos_bst_put  = []
tiempos_bst_get  = []

for n in ns:
    claves = random.sample(range(n * 10), n)

    # Sequential ST
    st_seq = SequentialST()
    t0 = time.perf_counter()
    for k in claves: st_seq.put(k, k)
    tiempos_seq_put.append((time.perf_counter() - t0) * 1000)

    t0 = time.perf_counter()
    for k in random.sample(claves, min(200, n)): st_seq.get(k)
    tiempos_seq_get.append((time.perf_counter() - t0) * 1000)

    # Binary Search ST
    st_bs = OrderedST(capacity=n + 10)
    t0 = time.perf_counter()
    for k in claves: st_bs.put(k, k)
    tiempos_bst_put.append((time.perf_counter() - t0) * 1000)

    t0 = time.perf_counter()
    for k in random.sample(claves, min(200, n)): st_bs.get(k)
    tiempos_bst_get.append((time.perf_counter() - t0) * 1000)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(ns, tiempos_seq_put, 'ro-', label='Sequential ST', linewidth=2)
axes[0].plot(ns, tiempos_bst_put, 'bs-', label='Binary Search ST', linewidth=2)
axes[0].set_xlabel('n'); axes[0].set_ylabel('Tiempo total (ms)')
axes[0].set_title('put: n inserciones'); axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(ns, tiempos_seq_get, 'ro-', label='Sequential ST', linewidth=2)
axes[1].plot(ns, tiempos_bst_get, 'bs-', label='Binary Search ST', linewidth=2)
axes[1].set_xlabel('n'); axes[1].set_ylabel('Tiempo total (ms)')
axes[1].set_title('get: 200 búsquedas'); axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.suptitle('Sequential Search vs Binary Search — Comparación Empírica',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"\n{'n':>6} {'Seq put':>10} {'BS put':>10} {'Seq get':>10} {'BS get':>10}")
print("-" * 50)
for i, n in enumerate(ns):
    print(f"{n:>6} {tiempos_seq_put[i]:>10.2f} {tiempos_bst_put[i]:>10.2f}"
          f" {tiempos_seq_get[i]:>10.2f} {tiempos_bst_get[i]:>10.2f}")

---
# Sección 6: Resumen (5 minutos)

## Lo que vimos hoy

| Concepto | Descripción |
|---------|-------------|
| **Symbol Table ADT** | Estructura clave-valor con operaciones put, get, delete |
| **Convenciones** | Sin claves null, sin duplicados, put(k,None) = delete |
| **Sequential Search** | Lista desordenada — get O(N), put O(N) |
| **Binary Search** | Arreglo ordenado — get O(log N), put O(N) |
| **rank(key)** | Núcleo de Binary Search ST — #claves menores que key |
| **Ordered ST API** | min, max, floor, ceiling, rank, select sobre claves comparables |
| **Ejemplo trenes** | floor = último tren antes de X; ceiling = próximo tren desde X |

## Lo que viene

El problema abierto es claro: necesitamos get **Y** put ambos en O(log N).  
La solución — **Binary Search Trees (BST)** — combina la estructura del árbol binario con la propiedad de orden de la Binary Search ST.

---
## Referencias

- Sedgewick & Wayne. *Algorithms*, 4ª ed., Sección 3.1 — Symbol Tables
- Sedgewick & Wayne. *Algorithms*, 4ª ed., Sección 3.2 — Binary Search Trees (próxima clase)

## 📚 Lecturas Recomendadas y Práctica

### Textbooks

| Libro | Edición | Capítulo | Tema |
|-------|---------|----------|------|
| Goodrich, Tamassia & Goldwasser (GTG) — *Data Structures and Algorithms in Python* | 1ª ed. | Cap. 10.1 | Mapas y el TDA Symbol Table |
| Miller & Ranum (M&R) — *Problem Solving with Algorithms and Data Structures Using Python* | 2011 | Cap. 5.5 | Búsqueda y diccionarios |
| Cormen et al. (CLRS) — *Introduction to Algorithms* | 4ª ed. | Cap. 11.1 | Tablas de acceso directo |

### Recursos gratuitos en línea

- 🌐 [VisuAlgo — Hash Table](https://visualgo.net/en/hashtable) — la alternativa no ordenada, para contrastar.
- 📄 [Python `dict`](https://docs.python.org/es/3/library/stdtypes.html#dict) — la Symbol Table de la biblioteca estándar.

### Práctica en Codeforces (soporta Python 3)

> 🔍 **Cómo filtrar:** ve a [codeforces.com/problemset](https://codeforces.com/problemset),
> escribe la etiqueta en **Tags** y ajusta **Rating**.

**Escala de dificultad orientativa para este curso:**

| Rating | Nivel | Descripción |
|--------|-------|-------------|
| 800 | ⭐ | Aplicación directa — la mayoría puede resolverlo |
| 1000–1200 | ⭐⭐ | Requiere una pequeña adaptación |
| 1300+ | ⭐⭐⭐ | Combina la idea con otra — desafío |

**Problemas recomendados para este tópico:**

| # | Problema | Rating | Por qué es útil |
|---|----------|--------|-----------------|
| 1 | [4C — Registration System](https://codeforces.com/problemset/problem/4/C) | ⭐⭐ 1300 | El caso de uso canónico: contar apariciones por clave |
| 2 | [1092B — Teams Forming](https://codeforces.com/problemset/problem/1092/B) | ⭐ 800 | Comparar el enfoque con diccionario y el de ordenar |
| 3 | [977C — Less or Equal](https://codeforces.com/problemset/problem/977/C) | ⭐⭐⭐ 1200 | Cuándo el orden importa y el diccionario no basta |

⚠️ Los dos primeros son el **mínimo esperado**. Los demás son desafío opcional.

## 🧪 Ejercicio 1: `rank` sobre arreglo ordenado ⭐

**Descripción:** implementa `rank(claves, clave)`: el número de claves **estrictamente
menores** que `clave` en un arreglo ordenado.

Es la operación central de una Symbol Table sobre arreglo ordenado: `get` es `rank` más una
comprobación, y `put` es `rank` más una inserción.

**Entrada:** `claves` (list ordenada, sin duplicados) y `clave`.
**Salida:** entero entre 0 y `len(claves)`.

**Ejemplo:**
```
Entrada: claves = [10, 20, 30, 40], clave = 30
Salida:  2        (hay dos claves menores: 10 y 20)

Entrada: claves = [10, 20, 30, 40], clave = 25
Salida:  2        (aunque 25 no esté)
```

**Restricciones:** debe funcionar aunque la clave no exista.
**Complejidad esperada:** O(log n)

In [ ]:
def rank(claves, clave):
    """
    Número de claves estrictamente menores que `clave`, por búsqueda binaria.

    Parámetros:
        claves (list): arreglo ordenado ascendentemente, sin duplicados
        clave: valor de referencia
    Retorna:
        int: cuántas claves son menores que `clave`
    """
    # Tu código aquí
    pass

In [ ]:
def verificar_ejercicio_1(fn):
    """Casos con y sin la clave presente, incluidos los extremos."""
    import random, time
    K=[10,20,30,40]
    casos = [
        ((K,30), 2, "clave presente"),
        ((K,25), 2, "clave ausente, en un hueco"),
        ((K,10), 0, "la menor"),
        ((K,40), 3, "la mayor"),
        ((K,5),  0, "por debajo del mínimo"),
        ((K,100),4, "por encima del máximo"),
        (([],7), 0, "arreglo vacío"),
        (([7],7),0, "un elemento, igual"),
    ]
    aprobados=0
    for (arr,c), esperado, desc in casos:
        t0=time.perf_counter()
        try:
            r=fn(list(arr),c); t1=time.perf_counter()
            if r==esperado:
                print(f"  ✅ {desc} ({(t1-t0)*1000:.2f}ms) -> {r}"); aprobados+=1
            else:
                print(f"  ❌ {desc}\n     Esperado: {esperado}\n     Obtenido: {r}")
        except Exception as e:
            print(f"  💥 {desc} — Error: {e}")
    random.seed(3); ok=True
    try:
        for _ in range(200):
            n=random.randint(0,40)
            arr=sorted(random.sample(range(200), n))
            c=random.randint(-5,205)
            if fn(list(arr),c)!=sum(1 for x in arr if x<c): ok=False; break
        print(f"  {'✅' if ok else '❌'} 200 pruebas aleatorias")
        aprobados+=ok
    except Exception as e:
        print(f"  💥 pruebas aleatorias — Error: {e}")
    total=len(casos)+1
    print(f"\n{'🎉 Todos los casos pasaron!' if aprobados==total else f'⚠️  {aprobados}/{total} casos correctos'}")

verificar_ejercicio_1(rank)

In [ ]:
# ═══════════════════════════════════════════════════
# SOLUCIÓN — Descomenta para ver después de intentarlo
# ═══════════════════════════════════════════════════

# def rank(claves, clave):
#     """Búsqueda binaria que converge al número de claves menores."""
#     lo, hi = 0, len(claves) - 1
#     while lo <= hi:
#         medio = (lo + hi) // 2
#         if clave < claves[medio]:
#             hi = medio - 1          # la respuesta está a la izquierda
#         elif clave > claves[medio]:
#             lo = medio + 1          # la respuesta está a la derecha
#         else:
#             return medio            # coincidencia exacta: hay `medio` menores
#     # Al salir, lo es exactamente el número de claves menores que `clave`
#     return lo
#     # Complejidad: O(log n) temporal, O(1) espacial

## 🧪 Ejercicio 2: Symbol Table sobre arreglo ordenado ⭐⭐

**Descripción:** implementa una Symbol Table que mantenga las claves **siempre ordenadas**,
con `put`, `get` y `contains`.

Es la implementación que hace `get` en $O(\log n)$ a costa de que `put` sea $O(n)$: insertar
en medio de un arreglo obliga a correr todo lo que viene después. Ese contraste es el
argumento para pasar a los árboles la semana siguiente.

**Métodos a implementar:**
- `put(clave, valor)` — inserta o **actualiza** si la clave ya existe
- `get(clave)` — devuelve el valor, o `None` si no está
- `__len__()` — número de pares almacenados
- `claves()` — lista de claves en orden ascendente

**Ejemplo:**
```python
st = STOrdenada()
st.put("pera", 3); st.put("ajo", 1); st.put("pera", 9)
st.claves()      -> ['ajo', 'pera']
st.get("pera")   -> 9
len(st)          -> 2
```

**Complejidad esperada:** `get` en O(log n), `put` en O(n)

> 💡 **Pista:** reutiliza `rank` del ejercicio anterior para hallar la posición de inserción.

In [ ]:
class STOrdenada:
    """
    Symbol Table sobre dos arreglos paralelos mantenidos en orden de clave.

    Complejidad:
        get: O(log n) por búsqueda binaria
        put: O(n) por el desplazamiento al insertar
    """

    def __init__(self):
        self._claves = []
        self._valores = []

    def put(self, clave, valor):
        """Inserta o actualiza. Complejidad: O(n)."""
        # Tu código aquí
        pass

    def get(self, clave):
        """Retorna el valor de la clave, o None. Complejidad: O(log n)."""
        # Tu código aquí
        pass

    def __contains__(self, clave):
        return self.get(clave) is not None

    def __len__(self):
        return len(self._claves)

    def claves(self):
        """Claves en orden ascendente."""
        return list(self._claves)

In [ ]:
def verificar_ejercicio_2(cls):
    """Comprueba orden, actualización, ausencias y consistencia con dict."""
    import random, time
    aprobados=0; total=6

    st=cls()
    for k,v in [("pera",3),("ajo",1),("pera",9)]:
        st.put(k,v)
    if st.claves()==["ajo","pera"]: print("  ✅ claves en orden y sin duplicados"); aprobados+=1
    else: print(f"  ❌ claves: esperado ['ajo','pera'], obtenido {st.claves()}")

    if st.get("pera")==9: print("  ✅ put actualiza el valor de una clave existente"); aprobados+=1
    else: print(f"  ❌ actualización: esperado 9, obtenido {st.get('pera')}")

    if len(st)==2: print("  ✅ __len__ no cuenta la clave repetida dos veces"); aprobados+=1
    else: print(f"  ❌ len: esperado 2, obtenido {len(st)}")

    if st.get("kiwi") is None: print("  ✅ get de clave ausente retorna None"); aprobados+=1
    else: print(f"  ❌ get ausente: esperado None, obtenido {st.get('kiwi')}")

    vacia=cls()
    if len(vacia)==0 and vacia.claves()==[] and vacia.get("x") is None:
        print("  ✅ tabla vacía se comporta bien"); aprobados+=1
    else: print("  ❌ tabla vacía")

    random.seed(4); ok=True
    try:
        for _ in range(60):
            ref={}; st2=cls()
            for _ in range(random.randint(0,40)):
                k=random.randint(0,25); v=random.randint(0,999)
                ref[k]=v; st2.put(k,v)
            if st2.claves()!=sorted(ref) or any(st2.get(k)!=ref[k] for k in ref):
                ok=False; break
        print(f"  {'✅' if ok else '❌'} 60 pruebas aleatorias contra un dict de referencia")
        aprobados+=ok
    except Exception as e:
        print(f"  💥 pruebas aleatorias — Error: {e}")
    print(f"\n{'🎉 Todos los casos pasaron!' if aprobados==total else f'⚠️  {aprobados}/{total} casos correctos'}")

verificar_ejercicio_2(STOrdenada)

In [ ]:
# ═══════════════════════════════════════════════════
# SOLUCIÓN — Descomenta para ver después de intentarlo
# ═══════════════════════════════════════════════════

# class STOrdenada:
#     """Symbol Table sobre arreglos paralelos ordenados por clave."""
#
#     def __init__(self):
#         self._claves = []
#         self._valores = []
#
#     def _rank(self, clave):
#         """Número de claves menores que `clave`. O(log n)."""
#         lo, hi = 0, len(self._claves) - 1
#         while lo <= hi:
#             medio = (lo + hi) // 2
#             if clave < self._claves[medio]:   hi = medio - 1
#             elif clave > self._claves[medio]: lo = medio + 1
#             else:                             return medio
#         return lo
#
#     def put(self, clave, valor):
#         """Inserta o actualiza. O(n) por el desplazamiento."""
#         i = self._rank(clave)
#         # Paso 1: si la clave ya está en la posición i, solo actualizamos
#         if i < len(self._claves) and self._claves[i] == clave:
#             self._valores[i] = valor
#             return
#         # Paso 2: si no, insertamos manteniendo el orden
#         self._claves.insert(i, clave)
#         self._valores.insert(i, valor)
#
#     def get(self, clave):
#         """O(log n)."""
#         i = self._rank(clave)
#         if i < len(self._claves) and self._claves[i] == clave:
#             return self._valores[i]
#         return None
#
#     def __contains__(self, clave): return self.get(clave) is not None
#     def __len__(self):             return len(self._claves)
#     def claves(self):              return list(self._claves)